# Hospital Resource Control: Q‑Learning + MCTS (Clean, Explained)

This is a refactored and annotated version of your original notebook. It adds:


- A clear section structure and table of contents

- Reproducibility controls

- Explanatory notes before major code blocks (imports, env setup, algorithms, training, evaluation)

- Output cleared, optional cells tagged


> Tip: Run cells top‑to‑bottom. The notebook is organized to read like a tutorial and run like a script.

## Table of Contents
1. [Environment & Reproducibility](#env)
2. [Imports & Configuration](#imports)
3. [Problem / Environment Setup](#problem)
4. [Algorithms](#algs)
   - [Q‑Learning](#qlearning)
   - [MCTS](#mcts)
5. [Training Loop](#train)
6. [Evaluation](#eval)
7. [Visualization](#viz)
8. [Next Steps](#next)


## 1) Environment & Reproducibility <a id='env'></a>
This cell sets deterministic seeds (when libraries are available) and makes plots readable.

In [ ]:
# Reproducibility & plotting defaults
import random, os
random.seed(42)
os.environ['PYTHONHASHSEED'] = '42'

try:
    import numpy as np
    np.random.seed(42)
except Exception:
    pass

try:
    import torch
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass

# Plot defaults (applies if matplotlib is used)
try:
    import matplotlib.pyplot as plt
    plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True})
except Exception:
    pass


## 2) Imports & Configuration <a id='imports'></a>
Centralized imports keep dependencies easy to scan. If a cell below installs packages, run it first.

In [ ]:
%matplotlib inline

#import gym
import matplotlib
import numpy as np
import itertools
from envs.Hospitalenv import Hostipal

import matplotlib.pyplot as plt
from envs import Plotting
from collections import defaultdict
import sys
import seaborn as sns
import timeit

matplotlib.style.use('ggplot')
import math

In [ ]:
lam=6 ## for generating request from possion distribution with Lambda = lam
beds=100

env=Hostipal(beds,lam)

## 4) Algorithms <a id='algs'></a>
Below are helper classes/functions used by Q‑Learning and MCTS.

In [ ]:
def mat(row,col,val):
    matrow=[]
    for i in range(row):
        matrow.append(val)
    mat=[]
    for j in range(col):
        mat.append(list(matrow))
    return mat


### 4.1 Q‑Learning <a id='qlearning'></a>
Tabular or function‑approximation Q‑Learning configuration and update rules live here.

In [ ]:
# Refactor tip: wrap blocks like this into functions with docstrings for reuse and testing.
Q_table=mat(env.m,env.m,0.5)
N_table=mat(env.m,env.m,1)
R_table=mat(env.m,env.m,0)


In [ ]:
def policy_MCTS(env,Q,N,s):
    max1=-100000000000
    a=None
    C=1.414
    
    
    for a in range(env.m-env.OccupiedBed):
        log_value=np.log(N[s][a])
        tar_value=log_value/sum(N[s])
        U=C*math.sqrt(tar_value)
        pi=U+Q[s][a]
        if pi>=max1:
            max1=pi
            action=a
    return action 
            
        

In [ ]:
def make_epsilon_greedy_policy(Q, epsilon, nA,env,N):
    """
    Creates an epsilon-greedy policy based on a given Q-function and epsilon.
    
    Args:
        Q: A dictionary that maps from state -> action-values.
            Each value is a numpy array of length nA (see below)
        epsilon: The probability to select a random action . float between 0 and 1.
        nA: Number of actions in the environment.
    
    Returns:
        A function that takes the observation as an argument and returns
        the probabilities for each action in the form of a numpy array of length nA.
    
    """
    def policy_fn(observation):
       
        A = np.ones(env.m-env.OccupiedBed, dtype=float) * epsilon / (env.m-env.OccupiedBed)
        
        #print("OccupiedBed",env.OccupiedBed)
        
        #print("A",A)
        
        best_action = policy_MCTS(env,Q,N,observation)
        A[best_action] += (1.0 - epsilon)
        return A
    return policy_fn


In [ ]:
# def make_epsilon_greedy_policy1(Q, epsilon, nA,env):
#     """
#     Creates an epsilon-greedy policy based on a given Q-function and epsilon.
    
#     Args:
#         Q: A dictionary that maps from state -> action-values.
#             Each value is a numpy array of length nA (see below)
#         epsilon: The probability to select a random action . float between 0 and 1.
#         nA: Number of actions in the environment.
    
#     Returns:
#         A function that takes the observation as an argument and returns
#         the probabilities for each action in the form of a numpy array of length nA.
    
#     """
#     def policy_fn(observation):
       
#         A = np.ones(env.m-env.OccupiedBed, dtype=float) * epsilon / (env.m-env.OccupiedBed)
        
#         #print("OccupiedBed",env.OccupiedBed)
        
#         #print("A",A)
        
#         best_action = np.argmax(Q[observation])
#         A[best_action] += (1.0 - epsilon)
#         return A
#     return policy_fn


In [ ]:
def q_learning(env, num_episodes, discount_factor=1.0, alpha=0.1, epsilon=0.1):
    """
    Q-Learning algorithm: Off-policy TD control. Finds the optimal greedy policy
    while following an epsilon-greedy policy
    
    Args:
        env: OpenAI environment.
        num_episodes: Number of episodes to run for.
        discount_factor: Gamma discount factor.
        alpha: TD learning rate.
        epsilon: Chance the sample a random action. Float betwen 0 and 1.
    
    Returns:
        A tuple (Q, episode_lengths).
        Q is the optimal action-value function, a dictionary mapping state -> action values.
        stats is an EpisodeStats object with two numpy arrays for episode_lengths and episode_rewards.
    """
    global Q_table
    global N_table
    global R_table
    nA = env.m 
    Q=Q_table
    N=N_table
    R=R_table
    #print(nA)
    
    # The final action-value function.
    # A nested dictionary that maps state -> (action -> action-value).
#Q = defaultdict(lambda: np.zeros(nA))
       

    #print(len(Q))
    # Keeps track of useful statistics
    stats = Plotting.EpisodeStats(
        episode_lengths=np.zeros(num_episodes),
        episode_rewards=np.zeros(num_episodes))    
    
    # The policy we're following
    policy = make_epsilon_greedy_policy(Q, epsilon, nA,env,N)
    
    for i_episode in range(num_episodes)  :
        
        epsilon /= (i_episode + 1)
        N=mat(env.m,env.m,1)
        
        # Print out which episode we're on, useful for debugging.
        if ((i_episode + 1) % 500 == 0):
            print("Episode {}/{}.".format(i_episode + 1, num_episodes),flush=True)
            sys.stdout.flush()
        
        #print("reset")
        state = env.reset()
        terminated = False
#         Q=MCTS(env,state)
        for t in itertools.count():

            #sys.stdout.write('\r'+str(t))
            
            request=max(1,env.request())
            #print("request",request)
            # sample the action from the epsilon greedy policy
            action = min(np.random.choice(env.m-env.OccupiedBed, p=policy(state)),request)
            #sys.stdout.write('actions --> \r'+ env.possibleaction[action])
            
            #print("action :",action)
            
            # Perform the action -> Get the reward and observe the next state
            #print("Before",env.getAgentRowAndColoumn(),"action",env.possibleaction[action])
            new_state, reward, terminated, old_state = env.step(action)

            
#             print("reward :",reward)
#             print("old state",old_state)
#             print("action",action)
#             print("new_state :",new_state)
#             print('-'*100)

            
            stats.episode_rewards[i_episode] += reward
            stats.episode_lengths[i_episode] = t

            # value that we should have got
            # The Q-learning target policy is a greedy one, hence the `max`
#             td_target = reward + discount_factor * max(Q[new_state])
#             td_error = td_target - Q[state][action]
            # Updating N
            N[state][action]+=1
            R[state][action]+=reward
            # Q-learning update
            Q[state][action] = (Q[state][action]+R[state][action]+max(Q[state]))/N[state][action]
            
            # update current state
            state = new_state
            
            if terminated:
                break
    return Q, stats

In [ ]:
# Refactor tip: wrap blocks like this into functions with docstrings for reuse and testing.
Q, stats = q_learning(env, 1000)

In [ ]:
dict_Q={}
for i , j in enumerate(Q):
    dict_Q[i]=np.array(j)
    
    

## 7) Visualization <a id='viz'></a>
Diagnostic plots: learning curves (reward per episode), queue sizes, utilization, etc.

In [ ]:
Plotting.plot_episode_stats(stats)

In [ ]:
#Plotting.plot_episode_stats(stats1)

In [ ]:
state_action=dict()
q_value=dict()
for i in dict_Q.keys():
    state_action[i]=np.argmax(Q[i])
    q_value[i]=np.max(Q[i])

In [ ]:
for i in sorted(state_action.keys()):
    print("i",i,"action",state_action[i]+1)

## 8) Next Steps <a id='next'></a>
- Add unit tests for your environment and Q/MCTS utilities (e.g., shape checks, deterministic rollouts with fixed seeds).
- Encapsulate algorithm hyperparameters in a config dict or dataclass.
- Log metrics to a CSV or TensorBoard for longer runs.
- Consider extracting large classes to `mcts.py` and `q_learning.py` modules and importing them here.